# Reproduce the search-complexity analysis

This notebook starts from one public CSV. It does not need the papers, teaching materials, or private source library.

One row is one observation: a numerical version of a scored, multi-issue negotiation matched to an observed agreement rate. The main analysis uses every row marked `main_eligible`. The stricter analysis keeps rows marked `strict_eligible` and uses the `strict_...` agreement evidence.

The regressions use HC3 robust standard errors. Continuous predictors are standardized within each fitted sample. The yes/no controls remain coded 0 or 1. Model 6 gives more weight to observations supported by more reported negotiation encounters.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Find the repository root whether Jupyter was opened from the root or a subfolder.
ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "search_complexity_dataset.csv").exists()
)
data = pd.read_csv(ROOT / "data" / "search_complexity_dataset.csv")

print(f"{len(data)} observations across {data['negotiation_scenario'].nunique()} named negotiation scenarios")
data.head(3)

## Build the main and strict datasets

The strict dataset uses the stricter outcome columns when they differ from the main coding. No values are statistically imputed.

In [ ]:
def as_bool(series):
    '''Read either Boolean values or the strings TRUE/FALSE safely.'''
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().eq("true")


main = data.loc[as_bool(data["main_eligible"])].copy()

strict = data.loc[as_bool(data["strict_eligible"])].copy()
strict["agreement_rate"] = strict["strict_agreement_rate"]
strict["agreements"] = strict["strict_agreements"]
strict["negotiations"] = strict["strict_negotiations"]

datasets = {"Main": main, "Strict": strict}

pd.DataFrame({
    "Dataset": datasets.keys(),
    "Observations": [len(frame) for frame in datasets.values()],
    "Named negotiation scenarios": [frame["negotiation_scenario"].nunique() for frame in datasets.values()],
})

## Regression functions and model definitions

- **M1:** search complexity
- **M2:** search complexity and the five basic controls
- **M3:** search complexity, bargaining-zone size, and the five basic controls
- **M4:** bargaining-zone size
- **M5:** bargaining-zone size and the five basic controls
- **M6:** Model 3 weighted by the reported number of negotiation encounters

The five controls are number of issues, number of parties, number of compatible issues, automated or AI counterpart, and non-unanimous agreement rule.

In [ ]:
BASIC_CONTROLS = [
    "number_of_issues",
    "number_of_parties",
    "number_of_compatible_issues",
    "automated_counterpart",
    "non_unanimous_rule",
]
CONTINUOUS = [
    "search_complexity",
    "bargaining_zone_share",
    "number_of_issues",
    "number_of_parties",
    "number_of_compatible_issues",
]
MODELS = {
    "M1": ["search_complexity"],
    "M2": ["search_complexity", *BASIC_CONTROLS],
    "M3": ["search_complexity", "bargaining_zone_share", *BASIC_CONTROLS],
    "M4": ["bargaining_zone_share"],
    "M5": ["bargaining_zone_share", *BASIC_CONTROLS],
}


def _betacf(a, b, x):
    '''Continued fraction used to calculate exact t-distribution probabilities.'''
    max_iter, eps, fpmin = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 - qab * x / qap
    d = fpmin if abs(d) < fpmin else d
    d = 1.0 / d
    h = d
    for m in range(1, max_iter + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        d = fpmin if abs(d) < fpmin else d
        c = 1.0 + aa / c
        c = fpmin if abs(c) < fpmin else c
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((qam + m2) * (qap + m2))
        d = 1.0 + aa * d
        d = fpmin if abs(d) < fpmin else d
        c = 1.0 + aa / c
        c = fpmin if abs(c) < fpmin else c
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return h


def _regularized_incomplete_beta(a, b, x):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    front = math.exp(
        math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
        + a * math.log(x) + b * math.log1p(-x)
    )
    if x < (a + 1.0) / (a + b + 2.0):
        return front * _betacf(a, b, x) / a
    return 1.0 - front * _betacf(b, a, 1.0 - x) / b


def two_sided_t_p(t_value, degrees_of_freedom):
    if not np.isfinite(t_value) or degrees_of_freedom <= 0:
        return np.nan
    x = degrees_of_freedom / (degrees_of_freedom + t_value * t_value)
    return float(_regularized_incomplete_beta(degrees_of_freedom / 2.0, 0.5, x))


def t_critical_975(degrees_of_freedom):
    low, high = 0.0, 20.0
    for _ in range(90):
        midpoint = (low + high) / 2.0
        if two_sided_t_p(midpoint, degrees_of_freedom) > 0.05:
            low = midpoint
        else:
            high = midpoint
    return (low + high) / 2.0


def zscore(series):
    '''Put a continuous variable on a mean-zero, one-standard-deviation scale.'''
    sd = series.std(ddof=0)
    return series * 0 if not np.isfinite(sd) or sd == 0 else (series - series.mean()) / sd


def fit(frame, predictors, weighted=False, inference=True):
    '''Fit OLS or negotiation-count-weighted least squares with HC3 errors.'''
    use = frame.dropna(subset=["agreement_rate", *predictors]).copy()
    x_frame = use[predictors].astype(float).copy()
    for column in predictors:
        if column in CONTINUOUS:
            x_frame[column] = zscore(x_frame[column])
    x_frame.insert(0, "const", 1.0)

    if weighted:
        weights = pd.to_numeric(use["negotiations"], errors="coerce")
        keep = weights.notna() & weights.gt(0)
        use = use.loc[keep].copy()
        x_frame = x_frame.loc[keep].copy()
        weights = weights.loc[keep].to_numpy(float)
    else:
        weights = None

    y = use["agreement_rate"].astype(float).to_numpy()
    x = x_frame.to_numpy(float)
    if weights is None:
        xt, yt = x, y
    else:
        root_w = np.sqrt(weights)
        xt, yt = x * root_w[:, None], y * root_w

    bread = np.linalg.pinv(xt.T @ xt)
    beta = bread @ xt.T @ yt
    fitted = x @ beta

    if inference:
        transformed_residual = yt - xt @ beta
        leverage = np.einsum("ij,jk,ik->i", xt, bread, xt)
        adjusted = transformed_residual / np.clip(1.0 - leverage, 1e-10, None)
        meat = xt.T @ ((adjusted * adjusted)[:, None] * xt)
        covariance = bread @ meat @ bread
        standard_error = np.sqrt(np.clip(np.diag(covariance), 0, None))
        t_value = np.divide(beta, standard_error, out=np.full_like(beta, np.nan), where=standard_error > 0)
        df_resid = max(len(y) - x.shape[1], 1)
        p_value = np.array([two_sided_t_p(value, df_resid) for value in t_value])
        critical = t_critical_975(df_resid)
        ci_low = beta - critical * standard_error
        ci_high = beta + critical * standard_error
    else:
        standard_error = p_value = ci_low = ci_high = np.full_like(beta, np.nan)

    if weights is None:
        center = y.mean()
        ss_total = np.sum((y - center) ** 2)
        ss_residual = np.sum((y - fitted) ** 2)
    else:
        center = np.average(y, weights=weights)
        ss_total = np.sum(weights * (y - center) ** 2)
        ss_residual = np.sum(weights * (y - fitted) ** 2)

    r_squared = 1 - ss_residual / ss_total
    names = list(x_frame.columns)
    terms = {
        name: {
            "beta": float(beta[index]),
            "se_hc3": float(standard_error[index]),
            "p": float(p_value[index]),
            "ci_low": float(ci_low[index]),
            "ci_high": float(ci_high[index]),
        }
        for index, name in enumerate(names)
    }
    return {
        "n": len(use),
        "terms": terms,
        "r_squared": float(r_squared),
        "adjusted_r_squared": float(1 - (1 - r_squared) * (len(y) - 1) / max(len(y) - x.shape[1], 1)),
    }


def hypothesis_statistics(frame):
    '''Return the four focal quantities used in the preregistered tests.'''
    fitted = {name: fit(frame, predictors) for name, predictors in MODELS.items()}
    return {
        "H1": {"value": fitted["M4"]["terms"]["bargaining_zone_share"]["beta"],
               "p": fitted["M4"]["terms"]["bargaining_zone_share"]["p"]},
        "H2": {"value": fitted["M1"]["terms"]["search_complexity"]["beta"],
               "p": fitted["M1"]["terms"]["search_complexity"]["p"]},
        "H3": {"value": fitted["M2"]["r_squared"] - fitted["M5"]["r_squared"], "p": np.nan},
        "H4": {"value": fitted["M3"]["terms"]["search_complexity"]["beta"],
               "p": fitted["M3"]["terms"]["search_complexity"]["p"]},
    }


def passes_full_sample(hypothesis, result):
    if hypothesis == "H3":
        return result["value"] > 0
    expected_positive = hypothesis == "H1"
    correct_direction = result["value"] > 0 if expected_positive else result["value"] < 0
    return correct_direction and result["p"] < 0.05

## Descriptive statistics and correlations

These tables describe the variables before the regressions. Agreement rate, bargaining-zone size, and search complexity remain on their original 0-to-1 scales here.

In [ ]:
KEY_VARIABLES = [
    "agreement_rate",
    "bargaining_zone_share",
    "search_complexity",
    *BASIC_CONTROLS,
]

descriptives = pd.concat(
    {
        name: frame[KEY_VARIABLES].agg(["count", "mean", "std", "min", "max"]).T
        for name, frame in datasets.items()
    },
    names=["Dataset", "Variable"],
)

display(descriptives.round(3))
display(main[KEY_VARIABLES].corr().round(3))

## Models 1-6 and the full-sample hypothesis tests

H1 is tested with M4. H2 is tested with M1. H3 compares the R-squared values of M2 and M5. H4 is tested with M3. M6 is a supplementary weighted version of M3.

In [ ]:
model_rows = []
full_sample_rows = []

for dataset_name, frame in datasets.items():
    fitted = {name: fit(frame, predictors) for name, predictors in MODELS.items()}
    fitted["M6"] = fit(frame, MODELS["M3"], weighted=True)

    for model_name, result in fitted.items():
        model_rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Observations": result["n"],
            "Search-complexity beta": result["terms"].get("search_complexity", {}).get("beta"),
            "Search-complexity HC3 SE": result["terms"].get("search_complexity", {}).get("se_hc3"),
            "Search-complexity p": result["terms"].get("search_complexity", {}).get("p"),
            "Bargaining-zone beta": result["terms"].get("bargaining_zone_share", {}).get("beta"),
            "Bargaining-zone HC3 SE": result["terms"].get("bargaining_zone_share", {}).get("se_hc3"),
            "Bargaining-zone p": result["terms"].get("bargaining_zone_share", {}).get("p"),
            "R-squared": result["r_squared"],
            "Adjusted R-squared": result["adjusted_r_squared"],
        })

    for hypothesis, result in hypothesis_statistics(frame).items():
        full_sample_rows.append({
            "Dataset": dataset_name,
            "Hypothesis": hypothesis,
            "Focal result": result["value"],
            "p": result["p"],
            "Full-sample rule passes": passes_full_sample(hypothesis, result),
        })

model_results = pd.DataFrame(model_rows)
full_sample_results = pd.DataFrame(full_sample_rows)

display(model_results.round(4))
display(full_sample_results.round(4))

## Preregistered robustness checks

The row bootstrap draws observations with replacement 5,000 times. It asks whether the expected result remains stable across many plausible samples of the same size. The 95% interval must stay entirely in the predicted direction.

The leave-one-observation-out check removes each observation once and refits the models. It asks whether one row determines the conclusion. For H1, H2, and H4, every refit must retain the predicted direction and remain statistically significant. For H3, every R-squared difference must remain positive.

The supplementary checks repeat the analysis after averaging observations within each named negotiation scenario, resampling whole named scenarios, and removing one named scenario at a time. They also apply row and scenario checks to weighted Model 6.

In [ ]:
BOOTSTRAP_DRAWS = 5000
RANDOM_SEED = 20260921


def quick_hypothesis_values(frame):
    '''Faster point estimates for bootstrap draws; no standard errors are needed.'''
    fitted = {name: fit(frame, predictors, inference=False) for name, predictors in MODELS.items()}
    return {
        "H1": fitted["M4"]["terms"]["bargaining_zone_share"]["beta"],
        "H2": fitted["M1"]["terms"]["search_complexity"]["beta"],
        "H3": fitted["M2"]["r_squared"] - fitted["M5"]["r_squared"],
        "H4": fitted["M3"]["terms"]["search_complexity"]["beta"],
    }


def draw_bootstrap(frame, draws, seed, whole_scenarios=False):
    rng = np.random.default_rng(seed)
    results = []
    if whole_scenarios:
        names = frame["negotiation_scenario"].drop_duplicates().tolist()
        groups = {name: frame.loc[frame["negotiation_scenario"] == name] for name in names}
    for _ in range(draws):
        if whole_scenarios:
            chosen = rng.choice(names, size=len(names), replace=True)
            sample = pd.concat([groups[name] for name in chosen], ignore_index=True)
        else:
            sample = frame.iloc[rng.integers(0, len(frame), len(frame))]
        try:
            results.append(quick_hypothesis_values(sample))
        except Exception:
            continue
    return pd.DataFrame(results)


def summarize_bootstrap(values, dataset_name, method):
    rows = []
    directions = {"H1": "positive", "H2": "negative", "H3": "positive", "H4": "negative"}
    for hypothesis, direction in directions.items():
        series = pd.to_numeric(values[hypothesis], errors="coerce").dropna()
        low, high = series.quantile([0.025, 0.975])
        rows.append({
            "Dataset": dataset_name,
            "Method": method,
            "Hypothesis": hypothesis,
            "Completed draws": len(series),
            "95% CI low": low,
            "95% CI high": high,
            "Interval rule passes": low > 0 if direction == "positive" else high < 0,
        })
    return rows


def leave_one_out(frame, whole_scenarios=False):
    rows = []
    if whole_scenarios:
        units = frame["negotiation_scenario"].drop_duplicates().tolist()
        samples = ((unit, frame.loc[frame["negotiation_scenario"] != unit]) for unit in units)
    else:
        samples = ((frame.iloc[index]["observation_id"], frame.drop(frame.index[index])) for index in range(len(frame)))
    for omitted, sample in samples:
        results = hypothesis_statistics(sample)
        rows.append({"Omitted": omitted, **{h: passes_full_sample(h, result) for h, result in results.items()}})
    return pd.DataFrame(rows)


def summarize_omission(values, dataset_name, method):
    return [{
        "Dataset": dataset_name,
        "Method": method,
        "Hypothesis": hypothesis,
        "Refits": len(values),
        "Refits passing rule": int(values[hypothesis].sum()),
        "All refits pass": bool(values[hypothesis].all()),
    } for hypothesis in ["H1", "H2", "H3", "H4"]]


def collapse_by_scenario(frame):
    '''Average rows within each named scenario, as specified in the preregistration.'''
    columns = ["agreement_rate", "bargaining_zone_share", "search_complexity", *BASIC_CONTROLS, "negotiations"]
    return frame.groupby("negotiation_scenario", as_index=False)[columns].mean()


def weighted_m6_beta(frame, inference=False):
    return fit(frame, MODELS["M3"], weighted=True, inference=inference)["terms"]["search_complexity"]


def bootstrap_m6(frame, draws, seed, whole_scenarios=False):
    rng = np.random.default_rng(seed)
    values = []
    if whole_scenarios:
        names = frame["negotiation_scenario"].drop_duplicates().tolist()
        groups = {name: frame.loc[frame["negotiation_scenario"] == name] for name in names}
    for _ in range(draws):
        if whole_scenarios:
            chosen = rng.choice(names, size=len(names), replace=True)
            sample = pd.concat([groups[name] for name in chosen], ignore_index=True)
        else:
            sample = frame.iloc[rng.integers(0, len(frame), len(frame))]
        try:
            values.append(weighted_m6_beta(sample)["beta"])
        except Exception:
            continue
    low, high = np.quantile(values, [0.025, 0.975])
    return {"Completed draws": len(values), "95% CI low": low, "95% CI high": high, "Interval entirely negative": high < 0}


def omit_for_m6(frame, whole_scenarios=False):
    passes = []
    if whole_scenarios:
        units = frame["negotiation_scenario"].drop_duplicates().tolist()
        samples = (frame.loc[frame["negotiation_scenario"] != unit] for unit in units)
    else:
        samples = (frame.drop(frame.index[index]) for index in range(len(frame)))
    for sample in samples:
        result = weighted_m6_beta(sample, inference=True)
        passes.append(result["beta"] < 0 and result["p"] < 0.05)
    return {"Refits": len(passes), "Refits negative and significant": sum(passes), "All refits pass": all(passes)}

In [ ]:
bootstrap_rows = []
omission_rows = []
scenario_model_rows = []
weighted_rows = []

for offset, (dataset_name, frame) in enumerate(datasets.items()):
    row_boot = draw_bootstrap(frame, BOOTSTRAP_DRAWS, RANDOM_SEED + offset, whole_scenarios=False)
    bootstrap_rows.extend(summarize_bootstrap(row_boot, dataset_name, "Resample observations"))

    scenario_boot = draw_bootstrap(frame, BOOTSTRAP_DRAWS, RANDOM_SEED + 10000 + offset, whole_scenarios=True)
    bootstrap_rows.extend(summarize_bootstrap(scenario_boot, dataset_name, "Resample named scenarios"))

    leave_row = leave_one_out(frame, whole_scenarios=False)
    omission_rows.extend(summarize_omission(leave_row, dataset_name, "Remove one observation"))

    leave_scenario = leave_one_out(frame, whole_scenarios=True)
    omission_rows.extend(summarize_omission(leave_scenario, dataset_name, "Remove one named scenario"))

    collapsed = collapse_by_scenario(frame)
    for model_name, predictors in MODELS.items():
        result = fit(collapsed, predictors)
        scenario_model_rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Named scenarios": result["n"],
            "Search-complexity beta": result["terms"].get("search_complexity", {}).get("beta"),
            "Search-complexity p": result["terms"].get("search_complexity", {}).get("p"),
            "Bargaining-zone beta": result["terms"].get("bargaining_zone_share", {}).get("beta"),
            "Bargaining-zone p": result["terms"].get("bargaining_zone_share", {}).get("p"),
            "R-squared": result["r_squared"],
        })

    weighted_full = weighted_m6_beta(frame, inference=True)
    for method, outcome in [
        ("Full weighted sample", {"Beta": weighted_full["beta"], "p": weighted_full["p"], "Passes": weighted_full["beta"] < 0 and weighted_full["p"] < 0.05}),
        ("Bootstrap observations", bootstrap_m6(frame, BOOTSTRAP_DRAWS, RANDOM_SEED + 20000 + offset, whole_scenarios=False)),
        ("Bootstrap named scenarios", bootstrap_m6(frame, BOOTSTRAP_DRAWS, RANDOM_SEED + 30000 + offset, whole_scenarios=True)),
        ("Remove one observation", omit_for_m6(frame, whole_scenarios=False)),
        ("Remove one named scenario", omit_for_m6(frame, whole_scenarios=True)),
    ]:
        weighted_rows.append({"Dataset": dataset_name, "Method": method, **outcome})

bootstrap_summary = pd.DataFrame(bootstrap_rows)
omission_summary = pd.DataFrame(omission_rows)
scenario_model_results = pd.DataFrame(scenario_model_rows)
weighted_m6_results = pd.DataFrame(weighted_rows)

display(bootstrap_summary.round(4))
display(omission_summary)
display(scenario_model_results.round(4))
display(weighted_m6_results.round(4))

## Confirmatory support summary

Under the preregistered rule, H1-H4 are supported only when the full-sample test, the 5,000-draw observation bootstrap, and the leave-one-observation-out check all pass. The named-scenario and Model 6 results are supplementary and do not replace this rule.

In [ ]:
support_rows = []
for dataset_name in datasets:
    for hypothesis in ["H1", "H2", "H3", "H4"]:
        full_pass = bool(full_sample_results.loc[
            (full_sample_results["Dataset"] == dataset_name) &
            (full_sample_results["Hypothesis"] == hypothesis),
            "Full-sample rule passes",
        ].iloc[0])
        boot_pass = bool(bootstrap_summary.loc[
            (bootstrap_summary["Dataset"] == dataset_name) &
            (bootstrap_summary["Method"] == "Resample observations") &
            (bootstrap_summary["Hypothesis"] == hypothesis),
            "Interval rule passes",
        ].iloc[0])
        omission_pass = bool(omission_summary.loc[
            (omission_summary["Dataset"] == dataset_name) &
            (omission_summary["Method"] == "Remove one observation") &
            (omission_summary["Hypothesis"] == hypothesis),
            "All refits pass",
        ].iloc[0])
        support_rows.append({
            "Dataset": dataset_name,
            "Hypothesis": hypothesis,
            "Full sample passes": full_pass,
            "Observation bootstrap passes": boot_pass,
            "Leave-one-observation-out passes": omission_pass,
            "Supported under preregistered rule": full_pass and boot_pass and omission_pass,
        })

pd.DataFrame(support_rows)